# **TP6 | Fonctions de hachage**

## 6.1. Hachage cryptographique

Une fonction de hachage cryptographique prend une liste d'octets de taille arbitraire et renvoie une liste d'octets de taille fixe.

Cette fonction doit être

1. déterministe : une entrée donne toujours la même sortie
2. chaotique : des données similaires donnent des hachés distants
3. uniforme : pour des données aléatoires, tous les hachés sont équiprobables

In [43]:
from hashlib import md5, sha256

### 6.1.a. Fonction `hash` de Python

À l'initialisation d'un script ou notebook, Python décide d'un sel aléatoire pour la fonction `hash`.

**TODO**  
Évaluer si la fonction `hash` de Python est déterministe. De même pour les hachages [MD5 et SHA-256 de `hashlib`](https://docs.python.org/3/library/hashlib.html#usage).

Pour ce faire, voici les résultats d'un hachage de `b"toto"` sur ma machine 

| algorithme | sortie |
| ---------- | ------ |
| `hash`     | 7220995997918355361 (int) |
| MD5        | f71dbe52628a3f83a77ab494817525c6 (hex) |
| SHA256     | MfemXjFVhqwZi9eYtmKc5JA9CJlHbVdBqfMuLlIbamY= (b64) |

In [44]:
data = b"toto"
hash(data)

2400889440305821934

In [45]:
hash_md5 = md5(data).digest()
print(hash_md5.hex())
#pareil que hash_md5 = md5(data).hexdigest() mais en deux lignes au lieu d'une seule

f71dbe52628a3f83a77ab494817525c6


In [46]:
from base64 import b64encode
hash_sha256 = sha256(data).digest()
#hash_md5_b64 = b64encode(hash_md5)
print(b64encode(hash_sha256))

#ou
#hash_sha256 = sha256(data).hexdigest()

b'MfemXjFVhqwZi9eYtmKc5JA9CJlHbVdBqfMuLlIbamY='


### 6.1.b. Collision MD5

Il y a trois degrés de sécurité pour une fonction de hachage :

1. PR : impossibilité de trouver une pré-image à un hash aléatoire
2. 2PR : impossibilité de trouver une pré-image qui donne le même hash que des données aléatoires
3. CR : impossibilité de trouver deux pré-images avec le même hash

La résistance aux collisions (CR) est le critère le plus restrictif et le plus couramment utilisé.

Trouver et exploiter des collisions est un domaine de recherche actif, notamment pour la falsification de certificats de sécurité. En théorie, Windows n'utilise plus de MD5 depuis 2013, notamment suite au [malware Flame](https://fr.wikipedia.org/wiki/Flame_(ver_informatique)), mais des failles utilisant MD5 ont néanmoins été [trouvées récemment](https://cendyne.dev/posts/2023-09-01-spoofing-certificates-with-md5-collisions.html).

**TODO**  
Comparer les checksums de `message1.bin` et `message2.bin` avec MD5 et SHA-256 dans une cellule, puis dans le terminal (`md5sum` et `sha256sum` pour les systèmes Unix, ["Get-FileHash"](https://learn.microsoft.com/en-us/powershell/module/microsoft.powershell.utility/get-filehash) sous Windows, ou bien [avec OpenSSL](https://docs.openssl.org/master/man1/openssl-dgst/#examples)).

Faire de même avec les images `plane.jpg` et `ship.jpg`.

In [47]:
with open("message1.bin", "rb") as f: #rb : read binary
    msg1 = f.read()
    #msg1 = open("message1.bin", "rb").read() : déconseillé car on ne ferme pas le fichier après lecture
    #ou
    #f1 = open("message1.bin", "rb")
    #msg1 = f1.read()
    #f1.close() : déconseillé car on ne ferme pas le fichier après lecture si une exception est levée entre l'ouverture et la fermeture du fichier
    
with open("message2.bin", "rb") as f: #rb : read binary
    msg2 = f.read()
    
print("Message 1 : ", msg1)
print("Message 2 : ", msg2)

hash1_md5 = md5(msg1).digest() #digest() : renvoie le hash sous forme binaire, hexdigest() : renvoie le hash sous forme hexadécimale
hash2_md5 = md5(msg2).digest() #digest() : finit de traiter les données de mon flux
print("Hash message 1 (MD5) : ", hash1_md5.hex())
print("Hash message 2 (MD5) : ", hash2_md5.hex())
hash1_sha256 = sha256(msg1).digest()
hash2_sha256 = sha256(msg2).digest()
print("Hash message 1 (SHA256) : ", hash1_sha256.hex())
print("Hash message 2 (SHA256) : ", hash2_sha256.hex())


Message 1 :  b'M\xc9h\xff\x0e\xe3\\ \x95r\xd4w{r\x15\x87\xd3o\xa7\xb2\x1b\xdcV\xb7J=\xc0x>{\x95\x18\xaf\xbf\xa2\x00\xa8(K\xf3n\x8eKU\xb3_Bu\x93\xd8Igm\xa0\xd1U]\x83`\xfb_\x07\xfe\xa2'
Message 2 :  b'M\xc9h\xff\x0e\xe3\\ \x95r\xd4w{r\x15\x87\xd3o\xa7\xb2\x1b\xdcV\xb7J=\xc0x>{\x95\x18\xaf\xbf\xa2\x02\xa8(K\xf3n\x8eKU\xb3_Bu\x93\xd8Igm\xa0\xd1\xd5]\x83`\xfb_\x07\xfe\xa2'
Hash message 1 (MD5) :  008ee33a9d58b51cfeb425b0959121c9
Hash message 2 (MD5) :  008ee33a9d58b51cfeb425b0959121c9
Hash message 1 (SHA256) :  54bcb9a4fda31e4f254303e3959acd5e420ad18a80949d56a3000c3716fbd1a0
Hash message 2 (SHA256) :  90774a6455a2bdb7d106e533923ecbefe81392ca55bed0ce81cfab2c1a7f0afe


In [48]:
print("Collision MD5 : ")
!md5sum plane.jpg ship.jpg
#Ces deux fichiers différents sont perçus comme identiques via MD5. Ils ont le même hash MD5 mais pas le même hash SHA256, donc il y a une collision pour MD5 mais pas pour SHA256.

Collision MD5 : 
253dd04e87492e4fc3471de5e776bc3d  plane.jpg
253dd04e87492e4fc3471de5e776bc3d  ship.jpg


## 6.2. HMAC

*Rappel :* L'objectif de la cryptographie est d'assurer la confidentialité et l'intégrité. Le chiffrement assure seulement la confidentialité.

Pour assurer l'intégrité d'un message, on peut y attacher un **MAC** (*Message Authentication Code*). Cela permet de valider la source et la non-modification du message.

Si ce MAC est basé sur une fonction de hachage, on parle de **HMAC**.

### 6.2.a. Padding de clé

La clé $K$ est pré-traitée pour calculer un HMAC.

**TODO**  
Compléter la fonction ci-dessous pour qu'elle corresponde à [`computeBlockSizedKey` de Wikipédia](https://en.wikipedia.org/wiki/HMAC#Implementation).

Faire le test sur la clé `"key"`.

In [49]:
def key_pad(key: bytes, algo) -> bytes:
    bs = algo().block_size
    n = len(key)
    
    '''
    if n > bs:
        raise ValueError(f"Key length {n} exceeds block size {bs}") #on peut lever une exception si la clé est trop longue, mais ce n'est pas le but ici, on veut juste la hacher si elle est trop longue
    if n < bs:
        key += b'\x00' * (bs - n)
    '''
    
    if n > bs:
        key = algo(key).digest()
    if n < bs:
        pad_size = bs - n
        key = key + bytes(pad_size) #padding avec des zéros
    return key

key = bytes(range(65))
print("Clé de 16 octets : ")
print(key_pad(key[:16], md5).hex())
print("Clé de 64 octets : ")
print(key_pad(key[:64], md5).hex())
print("Clé de 65 octets : ")
print(key_pad(key[:65], md5).hex())

Clé de 16 octets : 
000102030405060708090a0b0c0d0e0f000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
Clé de 64 octets : 
000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f202122232425262728292a2b2c2d2e2f303132333435363738393a3b3c3d3e3f
Clé de 65 octets : 
8bd7053801c768420faf816fadba971c


### 6.2.b. Calcul de HMAC

Une fois la clé pré-traitée, on définit 

$ \displaystyle\quad\qquad \begin{cases} K_\mathtt{o} = K \oplus \mathtt{opad} \\ K_\mathtt{i} = K \oplus \mathtt{ipad} \end{cases}$

avec $\mathtt{ipad} = \mathtt{0x36}$ et $\mathtt{opad} = \mathtt{0x5c}$, répétés autant que nécessaire. L'étiquette d'un message $t$ est alors calculée avec la formule

$ \displaystyle\quad\qquad \mathrm{HMAC}(K, t) = H\bigl( K_\mathtt{o} ~\|~ H (K_\mathtt{i} \| t) \bigr) $

**Attention :** En pratique, le MAC est calculé sur les données [déjà chiffrées](https://fr.wikipedia.org/wiki/Chiffrement_authentifi%C3%A9#Chiffrement-puis-MAC_(Encrypt-then-MAC,_ou_EtM)), et il n'utilise pas la même clé que le chiffrement.

**TODO**  
Implémenter ce calcul de HMAC puis effectuer des tests avec [les exemples de Wikipédia](https://en.wikipedia.org/wiki/HMAC#Examples).

In [53]:
def hmac(key: bytes, tag: bytes, algo=md5):
    block_sized_key = key_pad(key, algo)
    
    o_key_pad = bytes((b ^ 0x5c) for b in block_sized_key) #outer padding ; ^ is the bitwise XOR operator, pour chaque octet de la clé, on met un masque XOR avec 0x5c
    i_key_pad = bytes((b ^ 0x36) for b in block_sized_key) #inner padding
    
    i_hash = algo(i_key_pad + tag).digest()
    o_hash = algo(o_key_pad + i_hash).digest()
    
    return o_hash


key = bytes("key", 'utf-8')
message = bytes("The quick brown fox jumps over the lazy dog", 'utf-8')
ref_md5 = "80070713463e7749b90c2dc24911e275"
ref_sha256 = "f7bc83f430538424b13298e6aa6fb143ef4d59a14946175997479dbc2d1a3cd8"
test_hmac_md5 = hmac(key, message, md5)
print("Test MD5 validé : ", ref_md5 == test_hmac_md5.hex())
test_hmac_sha256 = hmac(key, message, sha256)
print("Test SHA256 validé : ", ref_sha256 == test_hmac_sha256.hex())

Test MD5 validé :  True
Test SHA256 validé :  True


### 6.2.c. Comparaison avec OpenSSL

On peut aussi utiliser OpenSSL pour [calculer des MAC](https://docs.openssl.org/master/man1/openssl-mac/#examples).

**TODO**  
Exécuter la commande suivante et valider le résultat avec votre fonction.

```sh
openssl mac -digest MD5 \
    -macopt hexkey:000102030405060708090A0B0C0D0E0F \
    -in message1.bin HMAC
```

In [ ]:
# TODO

## 6.3. Hachage de mots de passe

La fonction de hachage Argon2 est spécialisée dans le chiffrement de mots de passe. 

Nous allons l'étudier brièvement et comparer son utilisation avec SHA-256.

### 6.3.a. Hachage Argon2

Cette fonction de hachage est incluse dans [un module Python](https://argon2-cffi.readthedocs.io/en/stable/).

Avec la fonction `hash`, la sortie est une concaténation (séparée par des `$`) de 
- l'algorithme
- la version
- les paramètres (coût mémoire `m`, coût en temps `t` et parallélisation `p`)
- le sel en Base64
- le haché en Base64

On peut les récupérer en utilisant la méthode `str.split("$")`.

**TODO**  
Chiffrer un mot de passe. En octets, quelle est la taille du sel ? Quelle est la taille du haché ?

In [ ]:
import argon2

# TODO

### 6.3.b. Vérification Argon2

**TODO**  
Vérifier que le résultat 

```txt
$argon2id$v=19$m=65536,t=3,p=4$All+Them+Witches$aIFMB1pUf+7vQ/Q0WdUdT9l5iotOCrOu0nGQLR2ANP0
```

correspond bien au mot de pase `P@5sword!`.

In [ ]:
pwd_test = "P@5sword!"
hash_test = "$argon2id$v=19$m=65536,t=3,p=4$QUxMIFRIRU0gV0lUQ0hFUw$7rDvATKFD+m+e3OpLwWdbICLuv7zbJYLqqetwmswVbk"

# TODO

### 6.3.c. Mots de passe fréquents

On a haché 3 mots de passe aléatoires dans [une liste des 207 mdp les plus communs](https://github.com/danielmiessler/SecLists/blob/master/Passwords/Common-Credentials/probable-v2_top-207.txt).

**TODO**  
Retrouver les mots de passe correspondants.

In [ ]:
with open("hashes-top-207.txt", "r") as f:
    hashes = f.read().splitlines()

with open("probable-v2_top-207.txt", "r") as f:
    passwords = f.read().splitlines()

# TODO

### 6.3.d. Comparaison avec SHA256

On fixe le mot de passe `"P@5sword!"` et le sel `"ALL THEM WITCHES"`.

**TODO**  
Avec la commande magique `%%timeit`, comparer le temps de calcul pour hacher un mdp avec Argon2 et avec SHA-256. Pour SHA-256, chiffrer la concaténation `sel + pwd`. Lequel serait le plus facile à *brute force* ?

In [ ]:
pwd = "P@5sword!"
salt = b"ALL THEM WITCHES"

# TODO

## 6.A. Pour aller plus loin



### A.1. Clés de session

La fonction HMAC peut servir de générateur aléatoire pour dériver des clés de session.

Un algorithme courant est [**PBKDF2**](https://www.rfc-editor.org/rfc/rfc2898#section-5.2). Cela permet de convertir un secret partagé (avec Diffie-Hellman) en clé de chiffrement.

**TODO**  
Vérifier l'implémentation de PBKDF de `hashlib` avec [les cas test standards](https://www.rfc-editor.org/rfc/rfc6070#section-2).

Comparer avec `openssl kdf`:

`openssl kdf -keylen 20 -kdfopt digest:SHA1 -kdfopt pass:password -kdfopt salt:salt -kdfopt iter:2 PBKDF2`

In [ ]:
# TODO

### A.2. AES GCM

Beaucoup de connexions sont sécurisés avec la suite cryptographique « TLS\_AES\_128\_GCM\_SHA256 ». C'est le cas par exemple de Wikipédia.

Le mode GCM (Galois Counter Mode) est un mode de chiffrement **authentifié**. En plus de l'IV (appelé *nonce*) de 96 bits, il renvoie un MAC.

Attention, le *nonce* ne doit [**jamais**](https://frereit.de/aes_gcm/) être réutilisé avec une clé donnée. Sinon il devient très facile de falsifier le MAC.

**TODO**  
Utiliser le module `cryptography` pour chiffrer un message avec AES-GCM. Est-ce qu'il y a besoin de padding ? Comment récupérer le MAC ?

Déchiffrer les données (en b64) `sMfhRF5NdNuS4ZipugVPho0UdW0sjF1yAuRg` et `sMfhRF5NdNuS4ZipugVPho0UdW0sjF1yAuRh` qui ont été chiffrées avec la clé `"ALL THEM WITCHES"` et le nonce `"KNOWER 4EVER"` (pas d'AAD).

*Remarque :* Ce chiffrement n'est malheureusement [pas disponible](https://crypto.stackexchange.com/questions/76177/how-to-encrypt-files-using-aes256-gcm-cipher-under-linux) avec OpenSSL en ligne de commande.

In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# TODO

### A.3. Construction de Merkle-Damgård

Le hachage MD5 est basé sur une fonction de compression $f$ qui prend deux blocs de 64 octets et n'en renvoie qu'un.

Le haché final est alors calculé en découpant les données $x$ en blocs de 64 octets $x = x_1 \| x_2 \| ... \| x_N$ (avec *padding*) et en posant

$\displaystyle\quad\qquad\begin{cases} H_0 = IV \text{~~de 64 octets}, \\ H_i = f(H_{i-1}, x_i) \end{cases}$

Le haché est alors $\mathrm{MD5}(x_1 \| x_2 \| ... \| x_N) = H_N$. Le diagramme suivant (source [Wikipédia](https://commons.wikimedia.org/wiki/File:Merkle-Damgard-padding-fr.svg?uselang=fr)) illustre cela :

![](Merkle-Damgard-padding-fr.svg)

Le hachage SHA2 (le plus répandu aujourd'hui) utilise la même construction, avec des tailles différentes.

**TODO**  
Comparer le haché de la liste vide avec l'IV de MD5

```txt
67 45 23 01 ef cd ab 89 98 ba dc fe 10 32 54 76
```

Comment expliquer cette différence ?

Que se passe-t-il si on rajoute un préfixe à notre collision ? Et un suffixe ?